# Training

## Setup stage

Mounting Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cloning git repository to access the codebase.

In [ ]:
!git clone -b feat/XXXXXXXX --single-branch https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

pull for fast updates, if needed

In [ ]:
!git pull

Copying dataset from the Google Drive to the current local disk of VM.

In [ ]:
from tqdm import tqdm

# 1. Copy zip from Drive to local VM
!cp /content/drive/MyDrive/ImageNetSubset.zip /content/

# 2. Unzip directly to /content/
!unzip -q /content/ImageNetSubset.zip -d /content/

# 3. Rename "ImageNetSubset" to "datasets" 
!mv /content/ImageNetSubset /content/xAI-proj-m-ws2526/datasets

# 4. (Optional) Remove the zip to save space
!rm /content/ImageNetSubset.zip

Setup the root directory for the project. IMPORTANT for module imports.

In [ ]:
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward to find the outermost folder containing common project markers."""
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent  # keep going to prefer the outermost match
    return root or start

# Dynamically get the name of the cloned repository if it exists
cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

# If the cloned repository exists as a subdirectory, change into it
if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

# Now, find the project root from within the repository (or its parent if already there)
ROOT = find_project_root(Path.cwd()).resolve()

# Ensure we are in the identified project root
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print("cwd:", Path.cwd())
print("root on sys.path:", str(ROOT) in sys.path)

---

#### Only GU-Windows-Pool

In [ ]:
# Initialize conda for PowerShell
# Do this in the TERMINAL
& "C:\ProgramData\anaconda3\Scripts\conda.exe" init powershell

In [ ]:
!conda create -n xai-proj python=3.11 -y

---

#### Requirements

In [7]:
# Install dependencies
!pip install -r experiments/bagging/requirements.txt --quiet

In [ ]:
import wandb

# Login to WandB - this will prompt you to enter your API key
wandb.login()

#### Windows only

In [ ]:
Use this path on Windows: C:\Users\ba081274\Downloads\ImageNetSubset\ImageNetSubset\

## Training Stage

In [ ]:
!python -m experiments.bagging.scripts.train --config experiments/bagging/configs/default.yaml

# Sweep Stage

In [12]:
!wandb sweep experiments/bagging/configs/sweep_learning_rate.yaml --project resnet

wandb: Creating sweep from: experiments/bagging/configs/sweep_learning_rate.yaml
wandb: Creating sweep with ID: d9ecbw9d
wandb: View sweep at: https://wandb.ai/shady-university-of-bamberg/resnet/sweeps/d9ecbw9d
wandb: Run sweep agent with: wandb agent shady-university-of-bamberg/resnet/d9ecbw9d


In [13]:
# add the agent id here and --count 50 for roundes
!wandb agent shady-university-of-bamberg/resnet/d9ecbw9d


wandb: Starting wandb agent 🕵️
2025-12-28 22:17:53,750 - wandb.wandb_agent - INFO - Running runs: []
2025-12-28 22:17:53,988 - wandb.wandb_agent - INFO - Agent received command: run
2025-12-28 22:17:53,989 - wandb.wandb_agent - INFO - Agent starting run with config:
	batch_size: 64
	config: experiments/bagging/configs/default.yaml
	lr: 0.053572732920351446
	model: resnet34
	seed: 0
2025-12-28 22:17:53,990 - wandb.wandb_agent - INFO - About to run command: /usr/bin/env python -m experiments.bagging.scripts.train --batch_size=64 --config=experiments/bagging/configs/default.yaml --lr=0.053572732920351446 --model=resnet34 --seed=0
2025-12-28 22:17:58,996 - wandb.wandb_agent - INFO - Running runs: ['vt3pgdxx']
RESNET34 Training on ImageNetSubset

Random seed: 0
Device: CPU

 Dataset Summary:
   Training samples: 13032
   Validation samples: 500
   Classes: ['binder', 'coffee_mug', 'computer_keyboard', 'mouse', 'notebook', 'remote_control', 'soup_bowl', 'teapot', 'toilet_tissue', 'wooden_spo

## Validation Stage

In [ ]:
# Run ensemble evaluation on a dataset
!python -m experiments.scripts.inference --evaluate --data-dir datasets --wandb

In [ ]:
# Upload a test image or use sample
!python -m experiments.scripts.inference --image datasets/test_image.jpg --show-individual

---

## Script for saving best models in Google Drive

In [14]:
# Define source and destination paths
# We check experiments/bagging/checkpoints (relative to project root) first
source_dir = Path("experiments/bagging/checkpoints")

# Check for fallback paths if the default doesn't exist (e.g., if strictly using /chechpoints)
if not source_dir.exists():
    if Path("/chechpoints").exists(): # Handling the specific path mentioned
        source_dir = Path("/chechpoints")
    elif Path("/checkpoints").exists(): # Handling potential typo correction
        source_dir = Path("/checkpoints")

# Destination folder on the mounted drive
# You can change "saved_checkpoints" to your preferred folder name
dest_dir = Path("/content/drive/MyDrive/saved_checkpoints")

print(f"Source Directory: {source_dir}")
print(f"Destination Directory: {dest_dir}")

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Copy .pth files
if source_dir.exists():
    pth_files = list(source_dir.glob("*.pth"))

    if not pth_files:
        print("No .pth files found in source directory.")
    else:
        print(f"Found {len(pth_files)} .pth files to copy.")

        for file_path in pth_files:
            try:
                shutil.copy2(file_path, dest_dir / file_path.name)
                print(f"Successfully copied: {file_path.name}")
            except Exception as e:
                print(f"Error copying {file_path.name}: {e}")
else:
    print(f"Source directory {source_dir} not found. Please check the path.")

Source Directory: experiments/bagging/checkpoints
Destination Directory: /content/drive/MyDrive/saved_checkpoints
Source directory experiments/bagging/checkpoints not found. Please check the path.
